# 02_pool_only_filtering.ipynbScene classification for the swimming-pool-only scope.Categories: **POOL**, **OUT_OF_SCOPE**, **UNCERTAIN**.Each tile now shows the original image with all YOLO bounding boxes and class IDs/names from the original label files. Numeric class names are displayed literally and are not interpreted.This notebook only classifies images and writes a CSV manifest. It does not modify, move, or delete any dataset file and does not create a filtered dataset.

In [ ]:
import csvimport zipfilefrom collections import Counterfrom io import BytesIOfrom pathlib import Pathtry:    from PIL import Image, ImageDraw    from IPython.display import display, Image as IPyImageexcept ImportError as exc:    raise ImportError('Pillow and IPython are required.') from excDATASET_DIR = Path('E:/lifeguardDron/DrowningDetectionTracking_v1')CLEANING_ROOT = Path('E:/lifeguardDron/DrowningDetection_v1_cleaning')OUTPUT_DIR = CLEANING_ROOT / '02_pool_only_filtering'MANIFEST_PATH = CLEANING_ROOT / 'scene_classification_manifest.csv'SPLITS = ('train', 'valid', 'test')IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}BATCH_SIZE = 24SHEET_COLUMNS = 6TILE_SIZE = 240CAPTION_HEIGHT = 88ALLOWED = {'POOL', 'OUT_OF_SCOPE', 'UNCERTAIN'}CLASS_NAMES = {    0: '3',    1: '4',    2: '5',    3: '6',    4: 'Out of Water',    5: 'drowning',    6: 'swimming',}OUTPUT_DIR.mkdir(parents=True, exist_ok=True)print('Dataset:', DATASET_DIR)print('Output:', OUTPUT_DIR)print('Manifest:', MANIFEST_PATH)

In [ ]:
def find_split_zip(split):    candidates = sorted(DATASET_DIR.glob(f'{split}*.zip'))    return candidates[0] if candidates else Nonedef list_split_images(split):    zp = find_split_zip(split)    if zp is not None:        with zipfile.ZipFile(zp) as zf:            return [                i.filename                for i in zf.infolist()                if not i.is_dir()                and '/images/' in i.filename                and Path(i.filename).suffix.lower() in IMAGE_EXTENSIONS            ]    d = DATASET_DIR / split / 'images'    if d.is_dir():        return [str(p) for p in d.iterdir() if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS]    return []def read_image_bytes(split, image_name):    zp = find_split_zip(split)    if zp is not None:        with zipfile.ZipFile(zp) as zf:            return zf.read(image_name)    return Path(image_name).read_bytes()def label_name_for(image_name):    if '/images/' in image_name:        return image_name.replace('/images/', '/labels/').rsplit('.', 1)[0] + '.txt'    p = Path(image_name)    parts = list(p.parts)    if 'images' in parts:        parts[parts.index('images')] = 'labels'    else:        parts[-1] = p.stem + '.txt'    return str(Path(*parts))def read_label_text(split, image_name):    label_name = label_name_for(image_name)    zp = find_split_zip(split)    if zp is not None:        with zipfile.ZipFile(zp) as zf:            try:                return zf.read(label_name).decode('utf-8', errors='replace')            except KeyError:                return None    try:        return Path(label_name).read_text(encoding='utf-8', errors='replace')    except OSError:        return Nonedef parse_label_text(text):    if text is None:        return [], 'missing', 0    lines = [l.strip() for l in text.splitlines() if l.strip()]    if not lines:        return [], 'empty', 0    boxes = []    malformed = 0    for line in lines:        t = line.split()        if len(t) != 5:            malformed += 1            continue        try:            cid = int(t[0])            x, y, w, h = (float(v) for v in t[1:])        except ValueError:            malformed += 1            continue        if cid not in CLASS_NAMES:            malformed += 1            continue        if not all(0.0 <= v <= 1.0 for v in (x, y, w, h)) or w <= 0 or h <= 0:            malformed += 1            continue        boxes.append((cid, x, y, w, h))    status = ('malformed=' + str(malformed)) if malformed else 'ok'    return boxes, status, malformeddef load_manifest():    if not MANIFEST_PATH.is_file():        return {}    out = {}    with MANIFEST_PATH.open(newline='', encoding='utf-8') as f:        for row in csv.DictReader(f):            out[int(row['index'])] = row['category']    return outdef save_manifest(manifest, entries):    with MANIFEST_PATH.open('w', newline='', encoding='utf-8') as f:        w = csv.writer(f)        w.writerow(['index', 'split', 'image_name', 'category'])        for idx, e in enumerate(entries):            w.writerow([idx, e['split'], e['image_name'], manifest.get(idx, '')])

In [ ]:
all_entries = []for split in SPLITS:    for name in sorted(list_split_images(split)):        all_entries.append({'split': split, 'image_name': name})manifest = load_manifest()current_start = 0classified = sum(1 for k in manifest if manifest[k] in ALLOWED)print('total images:', len(all_entries))print('classified:', classified)print('unclassified:', len(all_entries) - classified)

In [ ]:
def render_tile(idx, entry):    try:        data = read_image_bytes(entry['split'], entry['image_name'])        im = Image.open(BytesIO(data)).convert('RGB')    except Exception as exc:        tile = Image.new('RGB', (TILE_SIZE, TILE_SIZE + CAPTION_HEIGHT), (40, 0, 0))        ImageDraw.Draw(tile).text((4, 4), 'image error: ' + str(exc)[:70], fill=(255, 255, 255))        return tile    boxes, status, malformed = parse_label_text(read_label_text(entry['split'], entry['image_name']))    ow, oh = im.size    scale = min(TILE_SIZE / ow, TILE_SIZE / oh)    nw, nh = max(1, int(ow * scale)), max(1, int(oh * scale))    resample = getattr(Image, 'Resampling', Image).LANCZOS    im = im.resize((nw, nh), resample)    canvas = Image.new('RGB', (TILE_SIZE, TILE_SIZE), (0, 0, 0))    canvas.paste(im, (0, 0))    d = ImageDraw.Draw(canvas)    for cid, x, y, w, h in boxes:        x1 = (x - w / 2) * nw        y1 = (y - h / 2) * nh        x2 = (x + w / 2) * nw        y2 = (y + h / 2) * nh        d.rectangle([x1, y1, x2, y2], outline=(0, 255, 0), width=1)        d.text((max(0, x1), max(0, y1)), str(cid) + ':' + CLASS_NAMES[cid], fill=(255, 255, 0))    tile = Image.new('RGB', (TILE_SIZE, TILE_SIZE + CAPTION_HEIGHT), (20, 20, 20))    tile.paste(canvas, (0, 0))    td = ImageDraw.Draw(tile)    fname = Path(entry['image_name']).name    cat = manifest.get(idx, '')    line1 = '#' + str(idx) + ' ' + entry['split'] + ' ' + fname    line2 = 'annot: ' + status + ' | boxes=' + str(len(boxes))    line3 = ('cat=' + cat) if cat else 'UNCLASSIFIED'    td.text((4, TILE_SIZE + 4), line1[:90], fill=(255, 255, 255))    td.text((4, TILE_SIZE + 26), line2[:90], fill=(200, 255, 200))    td.text((4, TILE_SIZE + 46), line3[:90], fill=(255, 255, 0) if cat else (255, 120, 120))    return tilebatch = [i for i in range(current_start, min(current_start + BATCH_SIZE, len(all_entries)))]if not batch:    print('End reached. current_start =', current_start)else:    cols = SHEET_COLUMNS    rows = (len(batch) + cols - 1) // cols    sheet = Image.new('RGB', (cols * TILE_SIZE, rows * (TILE_SIZE + CAPTION_HEIGHT)), (8, 8, 8))    for k, idx in enumerate(batch):        x = (k % cols) * TILE_SIZE        y = (k // cols) * (TILE_SIZE + CAPTION_HEIGHT)        sheet.paste(render_tile(idx, all_entries[idx]), (x, y))    sheet_path = OUTPUT_DIR / f'batch_{current_start:06d}_{len(batch)}.png'    sheet.save(sheet_path)    print('batch:', batch[0], 'to', batch[-1], '| sheet:', sheet_path.name)    display(IPyImage(filename=str(sheet_path)))

### Assign categories for the current batch1. Run the display cell above to see the current batch with bounding boxes and class IDs.2. Edit the decisions cell below using the global index numbers shown on the tiles.3. Use only **POOL**, **OUT_OF_SCOPE**, or **UNCERTAIN**.4. Run the apply cell, then the counts cell.5. Use the next/previous cells to move between batches.Indices not in the decisions dict stay unclassified.Annotation status is shown in the caption. Malformed or missing labels are displayed with an explicit status and are not treated as scene evidence.

In [ ]:
decisions = {    # 0: 'POOL',    # 1: 'OUT_OF_SCOPE',    # 2: 'UNCERTAIN',}

In [ ]:
bad = []for idx, cat in decisions.items():    if idx < 0 or idx >= len(all_entries):        bad.append((idx, cat, 'index out of range'))    elif cat not in ALLOWED:        bad.append((idx, cat, 'invalid category'))    else:        manifest[idx] = catif bad:    print('NOT APPLIED. Fix these entries:')    for b in bad:        print(b)else:    save_manifest(manifest, all_entries)    print('saved', len(decisions), 'decisions to', MANIFEST_PATH)

In [ ]:
c = Counter()for idx in range(len(all_entries)):    cat = manifest.get(idx, '')    if cat in ALLOWED:        c[cat] += 1total = len(all_entries)classified = sum(c.values())print('total:', total, '| classified:', classified, '| unclassified:', total - classified)for cat in ('POOL', 'OUT_OF_SCOPE', 'UNCERTAIN'):    n = c.get(cat, 0)    print(cat + ':', n, '(' + format(100 * n / total, '.2f') + '%)')print()for split in SPLITS:    idxs = [i for i, e in enumerate(all_entries) if e['split'] == split]    sc = Counter(manifest.get(i) for i in idxs if manifest.get(i) in ALLOWED)    un = len(idxs) - sum(sc.values())    print(split, dict(sc), 'unclassified=', un)

In [ ]:
current_start = min(current_start + BATCH_SIZE, len(all_entries))print('current_start =', current_start, '-> rerun the display cell')

In [ ]:
current_start = max(current_start - BATCH_SIZE, 0)print('current_start =', current_start, '-> rerun the display cell')

### Stop hereThis notebook performs scene classification only.Do not create the filtered pool-only dataset yet.Continue reviewing batches until all images are classified, then verify the counts. No original images or labels are modified.